# Lab 6: code generation in a real project

Lab 6 budget: $0.30, or nothing beyond a Claude subscription

**Every other lab calls a model from a cell. This one does not.**

Scenario 2, code generation with Claude Code. The repository is real, the client is the
`claude` CLI in a terminal, and this notebook builds the working copy and holds the prompts
you type.

Four routes, in one session: direct execution, the tests-first loop, plan mode, and a fork.
Then the two files that outlive the window they were written in.

**Nothing in this notebook runs a model.** Run it after Chapter 19.

## 1. The project, and your working copy

`mycorp/` sits next to this notebook, checked in, so you can read it before a session touches
it: the refund path of the MyCorp online shop, seven modules and two decision records.

The cell below copies it and gives the copy a history, **two commits and a `lab-start` tag**,
so `git diff lab-start` shows everything you have done at any point. `REBUILD = True` throws
the copy away and starts again. That is the reset.

![Lab 6: code generation](../diagrams/lab-06-code-generation.png)

In [ ]:
import labkit

lab = labkit.start(model_env=None, credential="none")

import mycorp_lab

REBUILD = False          # True throws the working copy away, losing anything in it

REPO, git = mycorp_lab.build(lab, rebuild=REBUILD)

## 2. What the project tells Claude, and the one thing nobody decided

**`CLAUDE.md` is the first thing a session in this directory reads, and it is guidance rather
than enforced configuration.**

| Convention | What it decides later |
|---|---|
| A validator returns a message and never raises | Whether the first fix coerces the string or rejects it |
| The gateway is the only network boundary | Which functions can be tested without patching |
| Tests only means the tests only | Whether the tests-first commit is honest |

`.claude/settings.json` is the other half of what ships with the project, and that one is
enforced rather than advisory: a narrow allow list, so a session can edit, run the tests and
commit without a dialog in front of each one. It is checked in, so everyone gets the same
answer, and `git push` is denied in it.

Then the decision records. **ADR-0007 says in as many words that no decision has been taken**,
which is the whole reason section 5 goes to plan mode.

In [ ]:
mycorp_lab.project_docs(REPO)

## 3. Route by shape, before you type

**Write the route down before each task, because deciding afterwards is not deciding.** The
test is CALM: **c**omplex architecture, **a**lternative approaches, **l**arge file count,
**m**ulti-step exploration. Any one of them true means plan mode.

| Ticket | Shape | Route |
|---|---|---|
| TKT-0053, the string amount | One file, a trace that names the line | Direct execution |
| TKT-0051, the reason codes | Prose has already produced two readings | Tests first, from examples |
| TKT-0054, tell the customer | Two defensible designs, no decision taken | Plan mode |

The cell below is the failure you are about to hand over. **A stack trace that names a file, a
function and a line is the clearest signal there is that a task wants direct execution**, because
there is nothing to discover and nothing to choose between. Hand it over as it is rather than
describing it.

In [ ]:
print(mycorp_lab.pytest_tail(REPO, "tests/test_refund_amount.py"))

## 4. Direct execution, and the tests-first loop

**Two ways of starting to type: when you know what to do, and when you know what it should do
but not how.** Open a terminal beside this notebook, and keep this one session for the rest of
the lab.

```bash
cd workspace/mycorp
claude --session-id 6b1e0e4a-0000-4000-8000-000000000006
```

Pinning the id is worth the extra characters. Section 6 forks this session, and a fixed id
makes that one line to copy rather than a picker to read.

### One. Direct execution, because the trace names the line

> `tests/test_refund_amount.py` fails with `TypeError: '>' not supported between instances of
> 'str' and 'float'`, raised inside `RefundAmountValidator.check` in
> `shop/validators/refund_amount.py`. The portal-initiated refund path sends `refund_amount`
> as a JSON string, for example `"12.50"`. Read `shop/validators/__init__.py` first, then fix
> the validator so all three tests pass. Do not coerce the string to a number: the pattern in
> this project says a validator returns a message for a value it will not accept, and never
> raises. One file. No plan.

CALM has nothing true here, which is what makes it direct execution. The judgement the ticket
hides is that **coercing would also make the test pass**. `shop/validators/__init__.py` is what
says it is wrong.

```bash
git add -A && git commit -m "TKT-0053: return a message for a non-numeric refund amount"
```

### Two. Tests first, because the prose has already failed twice

`normalise_reason` raises `NotImplementedError`, and TKT-0051 describes the rules in prose
that has already produced two different readings. **Examples are the answer**, and one of them
has to be the case the prose keeps losing.

> `shop/reasons.py` has `normalise_reason` raising `NotImplementedError`. Here is what it has
> to do, as examples rather than prose:
>
>     "CUST_DAMAGED"        -> "damaged"
>     "other: cracked lid"  -> "other"
>     None                  -> "unknown"
>
> `other` is a reason the customer gave. `unknown` is the absence of one: the gateway omits
> the field entirely on a merchant-initiated refund. Write `tests/test_reasons.py`, one case
> per example plus the empty string and an unrecognised code. The tests only. No
> implementation.

```bash
python -m pytest -q
git add tests/ && git commit -m "TKT-0051: tests before the implementation"
```

**Red, and that is the point.** The commit goes in while the tests still fail, because the
order is the one thing here that cannot be reconstructed afterwards. The implementation commit
goes on top of it, and from then on the order is permanent.

In [ ]:
print(git("log", "--oneline"))
print()
print(git("diff", "--stat", "lab-start") or "nothing changed since lab-start yet")

## 5. Plan mode, because nobody has decided

**Plan mode has two justifications, and only one of them survives a small repository.** Blast
radius is the famous one, and it does not apply here: this project is seven modules and you
could read all of it. So say the other one out loud. You are not in plan mode because of size,
you are in plan mode because nobody has decided yet, and that does not get cheaper as the
repository gets smaller.

Shift+Tab cycles into plan mode. Read the indicator rather than counting presses, because the
stops it offers vary with your settings.

> Read `docs/adr/ADR-0007-refund-notifications.md`. It says plainly that no decision has been
> taken. Plan TKT-0054 both ways: the refunds service sending the notification itself on the
> way out of `send_refund`, and writing a row to `shop/outbox.py` for `notifications-worker`
> to drain. At most eight lines per design. For each one, give the failure mode when the
> notification path is down, and name the constraint in the ADR that decides it. Recommend one
> and say why. No edits.

**The reason it gives is the deliverable, not the design.** Nothing else happens in this
session, so leaving plan mode on costs you nothing: the fork in section 6 is a new process and
starts in the default mode again.

**Three more routes, written up here rather than demonstrated.** A read-only subagent, for when
discovery would flood your window: `.claude/agents/trace-gateway.md` ships with the project, so
asking for `the trace-gateway subagent` is only offered in a session started in this directory,
and `Explore` is the built-in equivalent. A subagent reads in its own context and returns a
summary, so the greps never land in your window. And the edit that matches twice:
`timeout_seconds = 5.0` appears identically in two methods of `shop/gateway.py`, so an `Edit`
anchored on that line fails. The recovery is a longer anchor or a Read and a Write. **What must
not happen is `replace_all`**, which changes both.

## 6. Fork, the notepad, and what is still true

**A session is a transcript on disk, not a memory.** That is what makes a fork cheap: it
replays the same transcript under a new id, so two designs start from one baseline and neither
disturbs the other.

The cell below prints the ids recorded for this working copy, and the fork command ready to
copy.

### One. Two designs from one baseline

> Take the design where the refunds service sends the notification itself. Write it to
> `notes/fork-a-service-sends.md` under three headings: the change, the failure mode, what a
> test would assert. Then update `state/manifest.json`: set `baseline_commit` to the output of
> `git rev-parse HEAD`, set `choose-the-owner` to `done`, and put the files you actually read
> in its `files_read`. Keep the shape exactly as it is. No code.

The same command again, from the same id, gives you design B in `notes/fork-b-outbox.md`.
**The comparison is the deliverable, not the code.** Re-run the cell below afterwards and the
new id is on disk, with the original untouched.

### Two. Stale by arithmetic, not by feel

`notes/investigation.md` is prose, for a person to re-read. `state/manifest.json` is not: it
records which files a phase actually read and the commit it started from, so **whether a
finding still stands is an intersection rather than a judgement**. The last two cells are a
colleague changing a file while you were away, and then the sum.

In [ ]:
mycorp_lab.sessions(REPO)
print()
mycorp_lab.show_memory(REPO)

In [ ]:
mycorp_lab.colleague_commit(REPO, git)
print()
mycorp_lab.stale_phases(REPO, git)

## What you built

| The diagram | Where it happened |
|---|---|
| The repository, and its CLAUDE.md | Sections 1 and 2 |
| Direct execution, for a small stack-trace fix | Section 4, step one |
| Tests first, red before the implementation | Section 4, step two |
| Plan mode, for a decision nobody has taken | Section 5 |
| Fork, two designs from one baseline | Section 6, step one |
| Durable summary, and evidence that survives | Section 6, step two, and the git history |

The diagram shows more than this lab demonstrates, and the rest is Chapter 19. **Context
compaction** is the one to read first: `/compact` takes directions after it, and bare
`/compact` is the failure mode, because compaction replaces the conversation with a summary
and deletes the messages behind it. **Resume** and **a fresh session carrying a summary you
injected** are the other two ways back in.

**The decision to carry out of here is two sentences long.** Given a task and a codebase, say
whether to plan first or start typing, and say it before you type. Given a session that has
outlived its window, say what has to be written down, because the window will not keep it.